In [22]:
import pickle
import numpy as np

QuestDic=pickle.load(open("A50773_Ficha4_Respostas.p",'rb'))

# 1.a

In [23]:
# resolve_1a.py
import pickle
import numpy as np

# Carregar ficheiro (ajusta o path se necessário)
path = "A50773_Q001_data.p"
with open(path, "rb") as f:
    data = pickle.load(f)

# Verificar chaves
print("Chaves:", list(data.keys()))

# Ler X e y e normalizar formatos
X = np.asarray(data["X"])   # aparentemente shape (2, N)
y = np.asarray(data["y"])   # shape (N,)

# Se X for 2xN, transpor para N x 2
if X.shape[0] == 2 and X.shape[1] == y.shape[0]:
    X = X.T  # agora X tem shape (N,2)
elif X.shape[1] == 2 and X.shape[0] == y.shape[0]:
    # já está N x 2
    pass
else:
    raise ValueError(f"Formato inesperado de X: {X.shape}, y: {y.shape}")

# Converter y de {0,1} para {-1,+1} se necessário
unique_y = np.unique(y)
if set(unique_y) <= {0, 1}:
    y_desired = np.where(y == 0, -1.0, 1.0)
elif set(unique_y) <= {-1, 1}:
    y_desired = y.astype(float)
else:
    raise ValueError(f"Valores inesperados em y: {unique_y}")

N = X.shape[0]
print(f"N = {N}, X shape = {X.shape}, y unique = {unique_y}")

# Construir matriz de desenho com bias: X_design shape (N, 3)
X_design = np.hstack([np.ones((N,1)), X])  # colunas: [1, x1, x2]

# Calcular w_MSE por mínimos quadrados: w = (X^T X)^{-1} X^T y
# usar pseudo-inversa para estabilidade:
w, *_ = np.linalg.lstsq(X_design, y_desired, rcond=None)
w = w.flatten()  # [w0, w1, w2]

print("\nVetor de pesos wMSE (w0,w1,w2):")
print(w)

# Previsões contínuas e classificação por sinal
y_hat = X_design.dot(w)         # valores reais preditos
y_pred_class = np.where(y_hat >= 0, 1.0, -1.0)

# Erro absoluto médio (entre y desejado e y_hat)
MAE = np.mean(np.abs(y_desired - y_hat))

# Contagem de acertos na classe pi0 (classe negativa: y_desired == -1)
mask_pi0 = (y_desired == -1.0)
acertos_pi0 = np.sum((y_pred_class[mask_pi0] == -1.0))

# Informações adicionais
total_pi0 = np.sum(mask_pi0)
total_acertos = np.sum(y_pred_class == y_desired)

print(f"\nErro absoluto médio (MAE) = {MAE:.6f}")
print(f"Total pontos em π0 = {total_pi0}")
print(f"Número de acertos na classe π0 = {acertos_pi0}")
print(f"Número total de acertos = {total_acertos}")

# Comparar com alternativas do enunciado (com tolerâncias)
cond_i = np.isclose(MAE, 0.929, atol=1e-3)       # i. MAE == 0.929
cond_ii = (acertos_pi0 == 485)                  # ii. acertos π0 == 485

print("\nVerificação alternativas:")
print("i) MAE == 0.929 ->", cond_i)
print("ii) Nº acertos na π0 == 485 ->", cond_ii)

if cond_i and cond_ii:
    print("Resposta correcta: iii) Todas as respostas anteriores.")
elif cond_i and not cond_ii:
    print("Resposta correcta: i) apenas.")
elif (not cond_i) and cond_ii:
    print("Resposta correcta: ii) apenas.")
else:
    print("Resposta correcta: iv) Nenhuma das respostas anteriores.")


Chaves: ['y', 'X']
N = 2033, X shape = (2033, 2), y unique = [0. 1.]

Vetor de pesos wMSE (w0,w1,w2):
[ 0.15001065  0.08328478 -0.23408846]

Erro absoluto médio (MAE) = 0.611454
Total pontos em π0 = 573
Número de acertos na classe π0 = 485
Número total de acertos = 1945

Verificação alternativas:
i) MAE == 0.929 -> False
ii) Nº acertos na π0 == 485 -> True
Resposta correcta: ii) apenas.


# 1.b

In [25]:
import pickle
import numpy as np

# ---- nome do ficheiro ----
fname = "A50773_Q001_data.p"

with open(fname, "rb") as f:
    D = pickle.load(f)

X = D["X"]
y = D["y"]

# Corrigir orientação se necessário
if X.shape[1] == y.shape[0]:
    X = X.T

N = X.shape[0]
X_design = np.hstack([np.ones((N,1)), X])

# Vetor de pesos dado no enunciado
w = np.array([0.00, 0.96, -0.28])

# Previsões
y_pred = np.where(X_design @ w >= 0, 1, -1)

# Matriz de confusão
TP = np.sum((y == 1)  & (y_pred == 1))
TN = np.sum((y == -1) & (y_pred == -1))
FP = np.sum((y == -1) & (y_pred == 1))
FN = np.sum((y == 1)  & (y_pred == -1))

# Taxa de falsos alarmes
TFA = FP / np.sum(y == -1)

# Resultados importantes
acertos_pi1 = TP
acertos_pi0 = TN
acertos_totais = TP + TN

print("Taxa de falsos alarmes =", TFA)
print("Acertos π1 =", acertos_pi1)
print("Acertos π0 =", acertos_pi0)
print("Acertos totais =", acertos_totais)

# Comparação com as alternativas:
opt_i  = np.isclose(TFA, 0.412, atol=1e-3)
opt_ii = (acertos_pi1 == 1216)
opt_iii = (acertos_totais == 1549)
opt_iv = (acertos_pi0 == 338)

print("\nAlternativas verdadeiras:")
print("i  ->", opt_i)
print("ii ->", opt_ii)
print("iii->", opt_iii)
print("iv ->", opt_iv)


Taxa de falsos alarmes = nan
Acertos π1 = 1216
Acertos π0 = 0
Acertos totais = 1216

Alternativas verdadeiras:
i  -> False
ii -> True
iii-> False
iv -> False


C:\Users\Rafael\AppData\Local\Temp\ipykernel_10560\518587002.py:33: RuntimeWarning: invalid value encountered in scalar divide
  TFA = FP / np.sum(y == -1)


# 2.a

In [26]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import PolynomialFeatures

# carregar conjunto diabetes
data = load_diabetes()
X = data.data

n_features = X.shape[1]
print("Número de atributos originais:", n_features)

# ------------------------------
# Grau 2
# ------------------------------
poly2 = PolynomialFeatures(degree=2, include_bias=True)
X2 = poly2.fit_transform(X)
num_coef_2 = X2.shape[1]

print("Número de coeficientes grau 2:", num_coef_2)

# ------------------------------
# Grau 3
# ------------------------------
poly3 = PolynomialFeatures(degree=3, include_bias=True)
X3 = poly3.fit_transform(X)
num_coef_3 = X3.shape[1]

print("Número de coeficientes grau 3:", num_coef_3)


Número de atributos originais: 10
Número de coeficientes grau 2: 66
Número de coeficientes grau 3: 286


# 2.b

In [27]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np

# ------------------------------
# 1) Carregar dados
# ------------------------------
data = load_diabetes()
X = data.data
y = data.target

# ------------------------------
# 2) Dividir em treino (259) e teste (restantes)
# ------------------------------
X_train = X[:259]
y_train = y[:259]

X_test = X[259:]
y_test = y[259:]

# ------------------------------
# 3) Criar expansão polinomial de grau 2
# ------------------------------
poly = PolynomialFeatures(degree=2, include_bias=True)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# ------------------------------
# 4) Ajustar modelo MSE (regressão linear)
# ------------------------------
model = LinearRegression()
model.fit(X_train_poly, y_train)

# ------------------------------
# 5) Previsões e MAE
# ------------------------------
y_pred_train = model.predict(X_train_poly)
y_pred_test = model.predict(X_test_poly)

MAE_train = mean_absolute_error(y_train, y_pred_train)
MAE_test = mean_absolute_error(y_test, y_pred_test)

print("MAE treino =", MAE_train)
print("MAE teste  =", MAE_test)

# ------------------------------
# 6) Comparar com as alternativas
# ------------------------------
opt_i  = np.isclose(MAE_test, 45.82, atol=0.1)
opt_ii = np.isclose(MAE_train, 65.81, atol=0.1)
opt_iii = opt_i and opt_ii
opt_iv = not(opt_i or opt_ii)

print("\nAlternativas verdadeiras:")
print("i  ->", opt_i)
print("ii ->", opt_ii)
print("iii->", opt_iii)
print("iv ->", opt_iv)


MAE treino = 36.90561361829358
MAE teste  = 45.81920283829374

Alternativas verdadeiras:
i  -> True
ii -> False
iii-> False
iv -> False


# 3.a

In [28]:
# codigo_ex3a.py
import pickle
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# --- Ajusta o caminho se necessário ---
path = "A50773_Q003_data.p"

# --- Carregar os dados (tenta pickle; se o teu ambiente tiver problemas com pickle,
#     carrega o ficheiro no teu computador e converte para .npz com numpy.savez) ---
with open(path, "rb") as f:
    data = pickle.load(f)

# Verifica as chaves disponíveis
print("Chaves do dicionário:", list(data.keys()))

# Usar as chaves corretas
# o utilizador disse que a chave chama-se 'folds'
x = np.asarray(data["x"]).reshape(-1, 1)    # Nx1
y = np.asarray(data["y"]).reshape(-1,)     # Nx
folds = np.asarray(data["folds"]).reshape(-1,)  # Nx

# Separar treino/teste conforme enunciado:
# Treino = fold 1 ; Teste = fold 0
x_train = x[folds == 1]
y_train = y[folds == 1]
x_test  = x[folds == 0]
y_test  = y[folds == 0]

print(f"Nº pontos (treino): {len(y_train)}, (teste): {len(y_test)}")

# Construir features polinomiais de 4ª ordem
poly = PolynomialFeatures(degree=4, include_bias=True)  # inclui coluna de 1s
X_train = poly.fit_transform(x_train)  # shape (n_train, 5)
X_test  = poly.transform(x_test)

# Ajustar regressão linear (Mínimos quadrados)
model = LinearRegression(fit_intercept=False)  # fit_intercept False porque já temos bias em X (include_bias=True)
model.fit(X_train, y_train)

# Obter coeficientes w0..w4 (ordem: 1, x, x^2, x^3, x^4)
w = model.coef_.copy()
w0 = w[0]
w1, w2, w3, w4 = w[1], w[2], w[3], w[4]

print("\nCoeficientes obtidos (w0..w4):")
print(f"w0 = {w0}")
print(f"w1 = {w1}")
print(f"w2 = {w2}")
print(f"w3 = {w3}")
print(f"w4 = {w4}")

# Avaliação no treino
y_train_pred = model.predict(X_train)
R2_train = r2_score(y_train, y_train_pred)
MAE_train = mean_absolute_error(y_train, y_train_pred)

print("\nMétricas (treino):")
print(f"R2 (treino) = {R2_train:.6f}")
print(f"Erro absoluto médio (MAE, treino) = {MAE_train:.6f}")

# Avaliação no teste (opcional)
y_test_pred = model.predict(X_test)
MAE_test = mean_absolute_error(y_test, y_test_pred)
print(f"Erro absoluto médio (MAE, teste) = {MAE_test:.6f}")

# --- Comparar com as alternativas do enunciado ---
w0_rounded = int(np.round(w0, 0))  # arredondado a 0 casas decimais
R2_train_rounded = np.round(R2_train, 2)  # arredondar a 2 casas para comparar com 0.96

print("\n--- Resumo para as alternativas ---")
print(f"w0 arredondado a 0 casas decimais: {w0_rounded}")
print(f"R2 (treino) arredondado a 2 casas: {R2_train_rounded}")

# Decidir qual alternativa está correcta:
cond_i = (w0_rounded == 10)
cond_ii = (np.isclose(R2_train_rounded, 0.96, atol=0.005) or R2_train_rounded == 0.96)

print("\nAnálise das alternativas:")
print("i) w0 arredondado == 10 ->", cond_i)
print("ii) R2 (treino) == 0.96 ->", cond_ii)

if cond_i and cond_ii:
    print("Resposta correcta: iii) Todas as respostas anteriores.")
elif cond_i and not cond_ii:
    print("Resposta correcta: i) apenas.")
elif (not cond_i) and cond_ii:
    print("Resposta correcta: ii) apenas.")
else:
    print("Resposta correcta: iv) Nenhuma das respostas anteriores.")


Chaves do dicionário: ['x', 'y', 'folds']
Nº pontos (treino): 102, (teste): 151

Coeficientes obtidos (w0..w4):
w0 = -4.826684318651357
w1 = 4.7619842313919225
w2 = -1.1083983183461419
w3 = 0.08824175043019639
w4 = -0.0022708923260879033

Métricas (treino):
R2 (treino) = 0.956285
Erro absoluto médio (MAE, treino) = 0.243400
Erro absoluto médio (MAE, teste) = 0.266203

--- Resumo para as alternativas ---
w0 arredondado a 0 casas decimais: -5
R2 (treino) arredondado a 2 casas: 0.96

Análise das alternativas:
i) w0 arredondado == 10 -> False
ii) R2 (treino) == 0.96 -> True
Resposta correcta: ii) apenas.


# 3.b

In [29]:
import pickle
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# --- Carregar os dados ---
with open("A50773_Q003_data.p", "rb") as f:
    data = pickle.load(f)

x = np.asarray(data["x"]).reshape(-1, 1)
y = np.asarray(data["y"]).reshape(-1,)
folds = np.asarray(data["folds"]).reshape(-1,)

# ================================
# 3.b — Treino = fold 0 ; Teste = fold 1
# ================================
x_train = x[folds == 0]
y_train = y[folds == 0]

x_test = x[folds == 1]
y_test = y[folds == 1]

print(f"Nº pontos treino: {len(y_train)}, Nº pontos teste: {len(y_test)}")

# ================================
# Regressão polinomial de 3ª ordem
# ================================
poly = PolynomialFeatures(degree=3, include_bias=True)
X_train = poly.fit_transform(x_train)
X_test  = poly.transform(x_test)

model = LinearRegression(fit_intercept=False)
model.fit(X_train, y_train)

# ================================
# Avaliação
# ================================
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

R2_train = r2_score(y_train, y_train_pred)
MAE_test = mean_absolute_error(y_test, y_test_pred)

print("\nResultados 3.b:")
print(f"MAE (teste) = {MAE_test:.6f}")
print(f"R2 (treino) = {R2_train:.6f}")

# ================================
# Verificação das alternativas
# ================================
cond_i = np.isclose(MAE_test, 0.54, atol=0.01)
cond_ii = np.isclose(np.round(R2_train, 2), 0.59, atol=0.01)

print("\nVerificação das alternativas:")
print("i)  MAE teste = 0.54  ->", cond_i)
print("ii) R2 treino = 0.59  ->", cond_ii)

if cond_i and cond_ii:
    print("\nResposta correta: iii) Todas as respostas anteriores.")
elif cond_i:
    print("\nResposta correta: i)")
elif cond_ii:
    print("\nResposta correta: ii)")
else:
    print("\nResposta correta: iv) Nenhuma das respostas anteriores.")


Nº pontos treino: 151, Nº pontos teste: 102

Resultados 3.b:
MAE (teste) = 0.541595
R2 (treino) = 0.835834

Verificação das alternativas:
i)  MAE teste = 0.54  -> True
ii) R2 treino = 0.59  -> False

Resposta correta: i)


# No Final

In [31]:
QuestDic['Q001'][0,:]=np.array([0,1,0,0]) # Rever
QuestDic['Q001'][1,:]=np.array([0,1,0,0]) # Feito
QuestDic['Q002'][0,:]=np.array([1,0,0,0]) # Feito
QuestDic['Q002'][1,:]=np.array([1,0,0,0]) # Feito
QuestDic['Q003'][0,:]=np.array([0,1,0,0]) # Feito
QuestDic['Q003'][1,:]=np.array([1,0,0,0]) # Feito

pickle.dump(QuestDic,open('A50773_Ficha4_Respostas.p','wb'))

QuestDic

{'Q001': array([[0., 1., 0., 0.],
        [0., 1., 0., 0.]]),
 'Q002': array([[1., 0., 0., 0.],
        [1., 0., 0., 0.]]),
 'Q003': array([[0., 1., 0., 0.],
        [1., 0., 0., 0.]]),
 'nome': 'Rafael Alexandre Besteiro Dias',
 'numero': 'A50773'}